# Customer Segmentation and RFM Analysis

A retail company wants to segment its customer base to improve targeted marketing and understand their buying behaviors.


In [54]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from summarytools import dfSummary

In [79]:
df = pd.read_csv("PBL5recommendationdata.csv", encoding="latin-1")

C:\Users\kosey\AppData\Local\Temp\ipykernel_7464\258051885.py:1: DtypeWarning: Columns (20,33,73,106,158) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("PBL5recommendationdata.csv", encoding="latin-1")


In [80]:
df.head(3)

,Customers.id,Customers.fname,Customers.lname,Customers.company,Customers.create_date,Customers.status,Customers.mailing,Customers.reminders,Customers.tax_exempt,Customers.account_id,Customers.sales_rep,Customers.rewards,Customers.profile_id,Customers.last_modified,Customers.customer_type,Orders.id,Orders.customer_id,Orders.fname,Orders.lname,Orders.company,Orders.order_number,Orders.reorder_id,Orders.external_source,Orders.external_id,Orders.currency,Orders.sales_rep,Orders.subtotal,Orders.tax,Orders.shipping,Orders.coupon_id,Orders.coupon_amount,Orders.gift_id,Orders.gift_amount,Orders.fee_name,Orders.fee_amount,Orders.discount_name,Orders.discount_amount,Orders.total,Orders.balance_due,Orders.shipping_carrier,Orders.shipping_method,Orders.shipping_trans,Orders.shipping_flags,Orders.weight,Orders.tracking,Orders.payment_status,Orders.payment_date,Orders.payment_user,Orders.payment_type,Orders.payment_method,Orders.payment_amount,Orders.purchase_order,Orders.payment_id,Orders.payment_code,Orders.payment_ref,Orders.status,Orders.placed_date,Orders.updated_date,Orders.shipped_date,Orders.comments,Orders.notes,Orders.registry_id,Orders.gift_message,Orders.website,Orders.mailing,Orders.flags,Orders.partial_ship,Orders.customer_type,Order_Items.id,Order_Items.parent,Order_Items.product_id,Order_Items.product_name,Order_Items.attributes,Order_Items.attribute_names,Order_Items.attribute_prices,Order_Items.qty,Order_Items.price,Order_Items.cost,Order_Items.registry_item,Order_Items.related_id,Order_Items.reorder_frequency,Order_Items.account_id,Order_Items.flags,Products.id,Products.status,Products.product_type,Products.template,Products.vendor,Products.import_id,Products.name,Products.display_name,Products.menu_name,Products.list_price,Products.price,Products.sale_price,Products.cost,Products.flags,Products.left_flag,Products.right_flag,Products.last_modified,Products.taxable,Products.shopping_gtin,Products.shopping_brand,Products.shopping_mpn,Products.shopping_gender,Products.shopping_color,Products.shopping_age,Products.shopping_flags,Products.amazon_asin,Products.amazon_type,Products.amazon_item_type,Products.amazon_price,Products.google_shopping_id,Products.google_shopping_type,Products.google_shopping_cat,Products.google_adwords,Products.shopping_cat,Products.shopping_type,Products.pricegrabber_cat,Products.shopzilla_cat,Products.thefind_cat,Products.quickbooks_id,Products.qb_edit_sequence,Products.price_break_type,Products.price_breaks,Products.short_description,Products.long_description,Products.websites,Products.video,Products.audio,Products.seo_title,Products.seo_description,Products.seo_keywords,Products.seo_header,Products.seo_footer,Products.seo_url,Products.seo_category,Products.unit,Products.packaging,Products.display_packaging,Products.multiple,Products.length,Products.width,Products.height,Products.rx,Products.latex,Products.upc,Products.msds_link,Products.msds_label,Products.lit_link,Products.lit_label,Products.hcpcs,Products.case_qty,Products.markup,Products.override_markup,Products.notes,Products.import_flags,Products.map_price,Products.features_title,Products.warranty,Products.hygienic,Products.default_quantity,Products.user_size,Products.assembly,Products.installation,Products.shipping_length,Products.shipping_width,Products.shipping_height,Products.shipping_weight,Products.handling_time,Products.rotation_link,Products.google_shopping_label,Products.product_option,Products.size,Products.material,Products.arm_style,Products.leg_style,Products.seat_size,Products.family_id,Products.saved_status,Products.freight_cost
0,797,Christy,Dill,Company0,1426018724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1437764306,0.0,3758,797,Christy,Dill,Company0,3758,NaN,NaN,NaN,USD,NaN,57.20,0.0,9.95,13.0,2.86,NaN,NaN,NaN,NaN,NaN,NaN,64.29,NaN,fedex,11|Ground,NaN,NaN,NaN,5.7204E+14,3.0,1.426019e+09,NaN,authorize.net,NaN,64.29,NaN,6993607863,510142,NaN,1,1426019099,1.438868e+09,1.426101e+09,NaN,Insured By Eye4Fraud,NaN,NaN,NaN,NaN,NaN,NaN,0.0,528

In [81]:
dfSummary(df)

No,Variable,Stats / Values,Freqs / (% of Valid),Graph,Missing
1,Customers.id[int64],Mean (sd) : 1796.5 (1065.7)min < med < max:3.0 < 1747.5 < 3736.0IQR (CV) : 1833.5 (1.7),"3,054 distinct values","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAKoAAABGCAYAAABc8A97AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAAsdJREFUeJzt3b9u2lAYhvHvQEjMnxohWthYOlbqwMjSO+jFdu2WIbmBDF06sUSRIgeEG4QxdEkqVYqKfY5deNHz24/x8NjG30HC7fd7A05d49gnABRxUefBnXNtM7sMOETDzHaeazf7/f454LNxQmoL1TnXHg6HX+M4Hvisz/O8labpx16v97PRaGRl1y+Xy8Q5941Yz0Odd9TLOI4Hs9nsud/vr8suns/ng7u7uw/T6fTHeDxOyqxdLBbRzc3N4PHx8dLMCPUM1ProNzPr9/vr0Wj0q+y6JEnaZmbdbtdrvZm1PdbgRPEyBQmECgm1P/pVBU4smDhUjFDfEDqxYOJQvYOhBtxZ4t1u1/JYV4k8z1tmFjvnfJbHvV5vNJvNlmUnFkwc6vHPUEPuLFmWRXmef1qv19dm5vPW7i1N01aWZZ8nk0mz2WyWHo29nvvV1dW1z8Qhz/PY/C8Svja84dAd1XsW+jIH7Wy322bA+XnZbDYXURR1ptPpuuwM1izs3EMvkqenp5Vz7ruZlV774ixDL/Qd1WcW+joHPSbfGWzIuYdcJPf39+9ub2+/TCaT9z6RmwWHfrKR8zJVE5+LJEmSdsiTIDT0U34JJNQTFPIk8A29ipfAOkd6hHqGjrHtXPdIj1BRFe8X7yJ3c0JFpXx/hGQH7uaEij9CN0nq3OAhVJhZdZskdW3wECrM7LibJEUQKv5yjE2SIvg9KiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQQKiQU+hv0xWIRlT3warWKzMzSNI0eHh46/3O96mernnfo+iJ9/QZCU4ws3MwU2AAAAABJRU5ErkJggg=="">",0(0.0%)
2,Customers.fname[object],1. John2. Lee3. Carol C4. Robert5. Emily6. Mary7. James8. Linda9. Jeffrey10. Mark11. other,"46 (1.1%)44 (1.0%)35 (0.8%)35 (0.8%)35 (0.8%)34 (0.8%)34 (0.8%)32 (0.8%)32 (0.8%)31 (0.7%)3,836 (91.5%)","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAJsAAAD+CAYAAAAtWHdlAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAA7lJREFUeJzt3EFOG1kYRtHflpVuJmVZllgGC2ARLDaLYAHsAyFq0KSZuAch8xjJF7VzzhypBlfPdul9bE6n00Bh+9UPwJ9DbGTERkZsZMRGRmxkxEZGbGTERmYzMzcz821m3k+n09sXPw9XbHc8Hh+WZTms6/qy2Wy+C45L2S7Lcri7u5tlWQ7z84SDi9jOzNzc3Pz71Q/C9fMDgYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIiI2M2MiIjYzYyGxnZt7e3v766gfh+m3XdX15enqadV1fZub9qx+I62WDQGbjX2ZRcbKRsa4iY11FxrqKjJe6ZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERkZsZAxeyBi8kHEtnIzBC5lfJ9s41bi03fF4fJiZMXbh0rbLshyMXSh4z0ZGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERmb3MXSZMXbhwlwLJ2PwQsZ3NjI+RslYV5GxriLjOxsZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERsZsZERGxnrKjKuhZOxriLjZCNj8ELG4IWMVx9kxEZGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGQMXsi4Fk7G4IWMk42MwQsZgxcyXn2QERsZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERsZsZERGxnrKjI2CGSsq8g42chYV5GxriLj1QcZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERsZsZERGxmDFzKuhZMxeCHjZCNj8ELG4IWMVx9kxEZGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGQMXsi4Fk7G4IWM72xkfIySsa4iY11Fxnc2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjIzYy1lVkXAsnY11FxslGxuCFjMELGa8+yIiNjNjIiI2M2MiIjYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIGLyQcS2cjMELmV8nmyvhnOv93E/D3fF4fPjYIMBvW9f15dyR1G5ZlsP9/f3bfr//ccmH43q8vr7+/fj4eHh+fv42M78f28zMfr//cXt7+8/Fno5rdHPuH3jPRkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERmY38/PKyFc/CP8fn+1lt67ry+Pj42E+cWWEP9fHUOqskZRr4XzW2dfCDV7I+IFARmxkxEZGbGTERkZsZMRGRmxkxEbmP42qMa0t07nkAAAAAElFTkSuQmCC"">",0(0.0%)
3,Customers.lname[object],1. Ceccotti2. Riggleman3. Johnson4. Smith5. Emerson6. Whitener7. Baxt8. Samuels9. Brown10. Williams11. other,"41 (1.0%)35 (0.8%)33 (0.8%)28 (0.7%)27 (0.6%)25 (0.6%)25 (0.6%)23 (0.5%)20 (0.5%)20 (0.5%)3,917 (93.4%)","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAJsAAAD+CAYAAAAtWHdlAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAA7pJREFUeJzt3D1OY2cYhuHXljUJzbEsSyyDBbAIFptFeAHsAyFOESY0TjFMM0VCfritONdVU5zi1mf76HvYnM/ngcL20g/A/4fYyIiNjNjIiI2M2MiIjYzYyIiNzGZmbmbmy8y8nc/n1ws/D1dsdzweH5ZlOazr+rzZbH4RHJ9luyzL4e7ubpZlOcy3Ew4+xXZm5ubm5rdLPwjXzw8EMmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjs52ZeX19/enSD8L1267r+vz4+Djruj7PzNulH4jrZYNAZuNfZlH5frKNU43Ptjsejw8zM5ZVfLbtsiwHyyoK3rORERsZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERsZsZHZvQ9dZoxd+GSuhZMxeCHjZCNj8EL

In [102]:
print("Column names:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

Column names:
 ['Customers.id', 'Customers.fname', 'Customers.lname', 'Customers.company', 'Customers.create_date', 'Customers.status', 'Customers.mailing', 'Customers.reminders', 'Customers.tax_exempt', 'Customers.account_id', 'Customers.sales_rep', 'Customers.rewards', 'Customers.profile_id', 'Customers.last_modified', 'Customers.customer_type', 'Orders.id', 'Orders.customer_id', 'Orders.fname', 'Orders.lname', 'Orders.company', 'Orders.order_number', 'Orders.reorder_id', 'Orders.external_source', 'Orders.external_id', 'Orders.currency', 'Orders.sales_rep', 'Orders.subtotal', 'Orders.tax', 'Orders.shipping', 'Orders.coupon_id', 'Orders.coupon_amount', 'Orders.gift_id', 'Orders.gift_amount', 'Orders.fee_name', 'Orders.fee_amount', 'Orders.discount_name', 'Orders.discount_amount', 'Orders.total', 'Orders.balance_due', 'Orders.shipping_carrier', 'Orders.shipping_method', 'Orders.shipping_trans', 'Orders.shipping_flags', 'Orders.weight', 'Orders.tracking', 'Orders.payment_status', 'Order

In [84]:
# Customers tablosu
customers_cols = [c for c in df.columns if c.startswith("Customers.")]
customers_df = df[customers_cols].copy()
customers_df.columns = [c.replace("Customers.", "") for c in customers_df.columns]
customers_df = customers_df.drop_duplicates(subset=["id"])  # id bazlı tekrarı önle
print(f"Customers tablosu oluşturuldu ({customers_df.shape})")

Customers tablosu oluşturuldu ((3054, 15))


In [85]:
# Orders tablosu
orders_cols = [c for c in df.columns if c.startswith("Orders.")]
orders_df = df[orders_cols].copy()
orders_df.columns = [c.replace("Orders.", "") for c in orders_df.columns]
orders_df = orders_df.drop_duplicates(subset=["id"])
print(f"Orders tablosu oluşturuldu ({orders_df.shape})")

Orders tablosu oluşturuldu ((3565, 53))


In [86]:
# Products tablosu
products_cols = [c for c in df.columns if c.startswith("Products.")]
products_df = df[products_cols].copy()
products_df.columns = [c.replace("Products.", "") for c in products_df.columns]
products_df = products_df.drop_duplicates(subset=["id"])
print(f"Products tablosu oluşturuldu ({products_df.shape})")

Products tablosu oluşturuldu ((1711, 98))


In [87]:
# SQLite veritabanını oluşturuyoruz
conn = sqlite3.connect("customer_segmentation.db")

In [88]:
customers_df.to_sql("Customers", conn, if_exists="replace", index=False)
orders_df.to_sql("Orders", conn, if_exists="replace", index=False)
order_items_df.to_sql("Order_Items", conn, if_exists="replace", index=False)
products_df.to_sql("Products", conn, if_exists="replace", index=False)

print("Tüm tablolar SQLite veritabanına kaydedildi.")

Tüm tablolar SQLite veritabanına kaydedildi.


In [90]:
# Tabloları SQLite'tan tekrar oku (doğrulama için) ---
customers_db = pd.read_sql_query("SELECT * FROM Customers", conn)
orders_db = pd.read_sql_query("SELECT * FROM Orders", conn)
order_items_db = pd.read_sql_query("SELECT * FROM Order_Items", conn)
products_db = pd.read_sql_query("SELECT * FROM Products", conn)

print("Tablolar SQLite'tan başarıyla yüklendi.")
print(f"Customers: {customers_db.shape}, Orders: {orders_db.shape}, Order_Items: {order_items_db.shape}, Products: {products_db.shape}")

Tablolar SQLite'tan başarıyla yüklendi.
Customers: (3054, 15), Orders: (3565, 53), Order_Items: (4194, 15), Products: (1711, 98)


In [91]:
# Merge (ilişkilendirme) işlemi ---
# ID ilişkileri:
# Orders.customer_id → Customers.id
# Order_Items.parent → Orders.id
# Order_Items.product_id → Products.id

merged_df = (
    orders_db
    .merge(customers_db, left_on="customer_id", right_on="id", how="left", suffixes=("_order", "_customer"))
    .merge(order_items_db, left_on="id_order", right_on="parent", how="left")
    .merge(products_db, left_on="product_id", right_on="id", how="left", suffixes=("", "_product"))
)

print("Merge işlemi tamamlandı.")
print("Birleştirilmiş veri boyutu:", merged_df.shape)

Merge işlemi tamamlandı.
Birleştirilmiş veri boyutu: (4194, 181)


In [101]:
# Gerekli kolonları filtreleyelim
analysis_cols = [
    "id_customer", "fname_customer", "lname_customer",
    "customer_type_customer", "placed_date", "total",
    "tax", "shipping", "payment_method", "status_order", "product_name",
    "price_product", "qty", "cost_product"
]

df_analysis = merged_df[analysis_cols].copy()

print("Analize uygun tablo boyutu:", df_analysis.shape)
df_analysis.head(3)

Analize uygun tablo boyutu: (4194, 14)


,id_customer,fname_customer,lname_customer,customer_type_customer,placed_date,total,tax,shipping,payment_method,status_order,product_name,price_product,qty,cost_product
0,797,Christy,Dill,0.0,1426019099,64.29,0.0,9.95,None,1,"Basic Rollators, Green",57.64,1,44.00
1,3,John,Smith,0.0,1386090455,29.99,NaN,9.99,None,1,"Urinary Drain Bags,0.000",10.29,4,1.87
2,3,John,Smith,0.0,1449603652,78.73,0.0,9.95,None,3,"SensiCare Nitrile Exam Gloves, Blue, XX-Large",68.78,1,52.50


In [94]:
df_analysis["placed_date"] = pd.to_datetime(df_analysis["placed_date"], unit="s", errors="coerce")

In [95]:
snapshot_date = df_analysis["placed_date"].max() + pd.Timedelta(days=1)

rfm = (
    df_analysis.groupby("id_customer")
    .agg({
        "placed_date": lambda x: (snapshot_date - x.max()).days,  # Recency
        "id_customer": "count",                                   # Frequency
        "total": "sum"                                            # Monetary
    })
    .rename(columns={"placed_date": "Recency", "id_customer": "Frequency", "total": "Monetary"})
)

rfm.reset_index(inplace=True)
print("RFM tablosu oluşturuldu:", rfm.shape)
rfm.head()

RFM tablosu oluşturuldu: (3054, 4)


,id_customer,Recency,Frequency,Monetary
0,3,160,2,108.72
1,4,888,1,29.55
2,5,615,3,124.99
3,7,872,1,49.14
4,8,858,1,69.70


In [96]:
# RFM verisini ölçekle
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

# 4 segmentli K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Segment"] = kmeans.fit_predict(rfm_scaled)

print("K-Means segmentasyonu tamamlandı.")
rfm.groupby("Segment").agg({"Recency":"mean", "Frequency":"mean", "Monetary":"mean"})

K-Means segmentasyonu tamamlandı.


,Recency,Frequency,Monetary
Segment,,,
0,508.166127,1.152104,122.261165
1,109.814383,1.229835,118.975695
2,69.538462,19.769231,7005.669231
3,163.964286,6.035714,2252.230357


In [97]:
customer_segments = (
    rfm.merge(customers_db, left_on="id_customer", right_on="id", how="left")
    [["id_customer", "fname", "lname", "company", "Recency", "Frequency", "Monetary", "Segment"]]
)

customer_segments.head(10)

,id_customer,fname,lname,company,Recency,Frequency,Monetary,Segment
0,3,John,Smith,Company1,160,2,108.72,1
1,4,James,Anderson,None,888,1,29.55,0
2,5,Abraham,Pollak,Company3,615,3,124.99,0
3,7,peggy,thompson,None,872,1,49.14,0
4,8,Randy,Pruss,None,858,1,69.70,0
5,10,Tommy,Smith,None,844,1,34.00,0
6,11,Mark,Tremble,None,844,1,34.00,0
7,12,Emely,Cooke,None,843,1,31.47,0
8,13,george,mcmillin,None,454,3,322.93,0
9,14,adrian,Cavitt,None,839,1,349.98,0


In [98]:
customer_segments.to_csv("customer_segments.csv", index=False)
print("Segmentasyon sonucu 'customer_segments.csv' olarak kaydedildi.")

Segmentasyon sonucu 'customer_segments.csv' olarak kaydedildi.
